# HeritageLens — Final Production Conversational RAG Application (V3)

This is the complete, restart-safe **Part C application notebook**.

Run this notebook **once from top to bottom in a fresh Google Colab runtime**.
It loads the completed leakage-controlled V2 classifier and reconnects the
existing 71-record Chroma database. It does **not** retrain the classifier,
rebuild embeddings, insert records, repeat evaluation, or modify the training
notebook.

## Before running

1. Open Google Colab.
2. In the left sidebar, open **Secrets** (key icon).
3. Add a secret named `GOOGLE_API_KEY`.
4. Paste a Gemini API key from Google AI Studio.
5. Enable notebook access for that secret.
6. Select **Runtime → Run all**.

## Final configuration

- Official classifier: `efficientnetb0_v2_phase2_best.keras`
- Input size: `224 × 224 × 3`
- Supported styles: Romanesque, Gothic, Tudor Revival, Georgian, Art Deco
- Chroma collection: `heritagelens_architecture_v2`
- Expected records: `71`
- Embeddings: `sentence-transformers/all-MiniLM-L6-v2`
- Reranker: `cross-encoder/ms-marco-MiniLM-L-6-v2`
- LLM: the first available stable model from `gemini-3.6-flash`,
  `gemini-3.5-flash`, and `gemini-2.5-flash`
- Conversation memory: LangChain message history, isolated per Gradio session

The workflow is:

`Upload image → V2 prediction → retrieve evidence → rerank evidence →
Gemini explanation → follow-up conversation with memory → cited sources`


In [1]:
# CELL 1 — INSTALL FINAL RUNTIME DEPENDENCIES

%pip install -q -U \
    "gradio==6.20.0" \
    "chromadb>=1,<2" \
    "langchain>=1,<2" \
    "langchain-core>=1,<2" \
    langchain-chroma \
    langchain-huggingface \
    "langchain-google-genai>=4,<5" \
    "google-genai>=1,<2" \
    sentence-transformers

print("✅ Final runtime dependencies are ready.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.7/793.7 kB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 128.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# CELL 2 — MOUNT DRIVE, LOAD THE API KEY, AND VALIDATE SAVED ASSETS

from pathlib import Path
import getpass
import os
import uuid

from google.colab import drive


drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path("/content/drive/MyDrive/HeritageLens")
MODEL_FOLDER = PROJECT_ROOT / "models"
RAG_ROOT = PROJECT_ROOT / "PartC_RAG"
VECTOR_DB_PATH = RAG_ROOT / "vector_database"

# IMPORTANT: this is the official leakage-controlled V2 model.
FINAL_MODEL_PATH = (
    MODEL_FOLDER / "efficientnetb0_v2_phase2_best.keras"
)

required_paths = {
    "HeritageLens project": PROJECT_ROOT,
    "official V2 classifier": FINAL_MODEL_PATH,
    "RAG folder": RAG_ROOT,
    "Chroma database": VECTOR_DB_PATH / "chroma.sqlite3",
}

missing_paths = [
    f"{label}: {path}"
    for label, path in required_paths.items()
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required saved assets are missing:\n"
        + "\n".join(missing_paths)
    )


# Prefer a protected Colab Secret. If it is unavailable, prompt securely.
google_api_key = None

try:
    from google.colab import userdata

    google_api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    google_api_key = None

if not google_api_key:
    google_api_key = getpass.getpass(
        "Enter your Google Gemini API key: "
    ).strip()

if not google_api_key:
    raise RuntimeError(
        "A GOOGLE_API_KEY is required for the conversational RAG system."
    )

os.environ["GOOGLE_API_KEY"] = google_api_key

print("✅ Google Drive connected")
print(f"✅ Project: {PROJECT_ROOT}")
print(f"✅ Official classifier: {FINAL_MODEL_PATH.name}")
print(f"✅ Existing vector database: {VECTOR_DB_PATH}")
print("✅ Gemini API key loaded securely")


Mounted at /content/drive
Enter your Google Gemini API key: ··········
✅ Google Drive connected
✅ Project: /content/drive/MyDrive/HeritageLens
✅ Official classifier: efficientnetb0_v2_phase2_best.keras
✅ Existing vector database: /content/drive/MyDrive/HeritageLens/PartC_RAG/vector_database
✅ Gemini API key loaded securely


In [3]:
# CELL 3 — LOAD THE V2 CLASSIFIER, EXISTING RAG DATABASE, RERANKER, AND LLM

import chromadb
import numpy as np
import tensorflow as tf
import torch

from google import genai
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder


CLASS_NAMES = [
    "romanesque",
    "gothic",
    "tudor_revival",
    "georgian",
    "art_deco",
]

DISPLAY_NAMES = {
    "romanesque": "Romanesque",
    "gothic": "Gothic",
    "tudor_revival": "Tudor Revival",
    "georgian": "Georgian",
    "art_deco": "Art Deco",
}

IMAGE_SIZE = (224, 224)
REQUIRED_COLLECTION = "heritagelens_architecture_v2"
EXPECTED_RECORD_COUNT = 71
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
PREFERRED_LLM_MODELS = [
    "gemini-3.6-flash",
    "gemini-3.5-flash",
    "gemini-2.5-flash",
]


# Load or safely reuse the official V2 classifier.
existing_model = globals().get("heritage_model")
loaded_model_path = globals().get("_HERITAGELENS_MODEL_PATH")

if (
    existing_model is not None
    and loaded_model_path == str(FINAL_MODEL_PATH)
    and existing_model.input_shape == (None, 224, 224, 3)
    and existing_model.output_shape == (None, 5)
):
    heritage_model = existing_model
    print("Reusing the official V2 classifier.")
else:
    tf.keras.backend.clear_session()
    heritage_model = tf.keras.models.load_model(
        FINAL_MODEL_PATH,
        compile=False,
    )
    _HERITAGELENS_MODEL_PATH = str(FINAL_MODEL_PATH)
    print("Loaded the official V2 classifier.")

if heritage_model.input_shape != (None, 224, 224, 3):
    raise ValueError(
        f"Unexpected classifier input shape: {heritage_model.input_shape}"
    )

if heritage_model.output_shape != (None, len(CLASS_NAMES)):
    raise ValueError(
        f"Unexpected classifier output shape: {heritage_model.output_shape}"
    )

classifier_model = heritage_model


# Reconnect the completed Chroma database without changing records.
chroma_client = chromadb.PersistentClient(
    path=str(VECTOR_DB_PATH)
)

collection_items = chroma_client.list_collections()
collection_names = [
    item if isinstance(item, str) else item.name
    for item in collection_items
]

if REQUIRED_COLLECTION not in collection_names:
    raise RuntimeError(
        f"Required collection not found: {REQUIRED_COLLECTION}. "
        f"Available collections: {collection_names}"
    )

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vector_database = Chroma(
    client=chroma_client,
    collection_name=REQUIRED_COLLECTION,
    embedding_function=embedding_model,
)

stored_count = vector_database._collection.count()

if stored_count != EXPECTED_RECORD_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_RECORD_COUNT} RAG records, "
        f"but found {stored_count}."
    )


# Load the cross-encoder used to rerank retrieved passages.
reranker_device = "cuda" if torch.cuda.is_available() else "cpu"
existing_reranker = globals().get("reranker")

if (
    existing_reranker is not None
    and globals().get("_HERITAGELENS_RERANKER_NAME")
    == RERANKER_MODEL_NAME
    and globals().get("_HERITAGELENS_RERANKER_DEVICE")
    == reranker_device
):
    reranker = existing_reranker
    print("Reusing the loaded reranker.")
else:
    reranker = CrossEncoder(
        RERANKER_MODEL_NAME,
        device=reranker_device,
    )
    _HERITAGELENS_RERANKER_NAME = RERANKER_MODEL_NAME
    _HERITAGELENS_RERANKER_DEVICE = reranker_device
    print("Loaded the reranker.")


# Resolve a text-generation model that is actually available to this API key.
# This prevents a notebook from launching successfully and then failing only
# when the first Gradio callback reaches an unavailable Gemini model.
gemini_client = genai.Client(
    api_key=google_api_key
)

try:
    available_model_names = {
        str(model.name).removeprefix("models/")
        for model in gemini_client.models.list()
        if getattr(model, "name", None)
    }
except Exception as error:
    raise RuntimeError(
        "Gemini could not list the models available to this API key. "
        "Check GOOGLE_API_KEY and its AI Studio access."
    ) from error

LLM_MODEL_NAME = next(
    (
        model_name
        for model_name in PREFERRED_LLM_MODELS
        if model_name in available_model_names
    ),
    None,
)

if LLM_MODEL_NAME is None:
    raise RuntimeError(
        "None of the supported Gemini text models is available to this "
        "API key. Available Gemini models include: "
        + ", ".join(
            sorted(
                name
                for name in available_model_names
                if name.startswith("gemini")
            )[:20]
        )
    )


# Connect the selected LangChain chat model.
llm = ChatGoogleGenerativeAI(
    model=LLM_MODEL_NAME,
    google_api_key=google_api_key,
    temperature=0.2,
    max_retries=2,
)


print("\nFINAL HERITAGELENS PIPELINE")
print("-" * 72)
print(f"Classifier: {FINAL_MODEL_PATH.name}")
print(f"Classifier input: {classifier_model.input_shape}")
print(f"Class order: {CLASS_NAMES}")
print(f"RAG collection: {REQUIRED_COLLECTION}")
print(f"RAG records: {stored_count}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Reranker: {RERANKER_MODEL_NAME}")
print(f"Reranker device: {reranker_device}")
print(f"LLM: {LLM_MODEL_NAME}")
print("\n✅ Official V2 classifier connected")
print("✅ Existing 71-record database connected")
print("✅ Embeddings and reranker connected")
print("✅ LangChain LLM connected")
print("✅ No training or database modification performed")


Loaded the official V2 classifier.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Loaded the reranker.

FINAL HERITAGELENS PIPELINE
------------------------------------------------------------------------
Classifier: efficientnetb0_v2_phase2_best.keras
Classifier input: (None, 224, 224, 3)
Class order: ['romanesque', 'gothic', 'tudor_revival', 'georgian', 'art_deco']
RAG collection: heritagelens_architecture_v2
RAG records: 71
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2
Reranker device: cuda
LLM: gemini-3.6-flash

✅ Official V2 classifier connected
✅ Existing 71-record database connected
✅ Embeddings and reranker connected
✅ LangChain LLM connected
✅ No training or database modification performed


In [4]:
# CELL 4 — OFFICIAL V2 IMAGE CLASSIFICATION

from pathlib import Path


def load_official_image_tensor(image_path):
    """
    Decode and resize with the same TensorFlow path used by the
    completed V2 evaluation workflow.
    """
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(
            f"Image not found: {image_path}"
        )

    image_bytes = tf.io.read_file(str(image_path))

    try:
        # Decode from the file bytes, not the temporary Gradio filename.
        # Colab uploads may use extensionless temporary paths.
        image_tensor = tf.io.decode_image(
            image_bytes,
            channels=3,
            expand_animations=False,
        )
    except Exception as error:
        raise ValueError(
            "The uploaded file could not be decoded as a valid image. "
            "Please use JPG, JPEG, PNG, BMP, or WEBP."
        ) from error

    image_tensor.set_shape([None, None, 3])

    image_tensor = tf.image.resize(
        image_tensor,
        IMAGE_SIZE,
        method="bilinear",
        antialias=False,
    )

    image_tensor = tf.cast(
        image_tensor,
        tf.float32,
    )

    return image_tensor


def classify_architecture_image(
    image_path,
    confidence_threshold=0.60,
    margin_threshold=0.15,
):
    image_tensor = load_official_image_tensor(
        image_path
    )

    model_batch = tf.expand_dims(
        image_tensor,
        axis=0,
    )

    probabilities = classifier_model(
        model_batch,
        training=False,
    ).numpy()[0]

    ranked_indices = np.argsort(
        probabilities
    )[::-1]

    top_3 = [
        {
            "style": CLASS_NAMES[int(index)],
            "display_style": DISPLAY_NAMES[
                CLASS_NAMES[int(index)]
            ],
            "confidence": float(
                probabilities[int(index)]
            ),
        }
        for index in ranked_indices[:3]
    ]

    confidence = top_3[0]["confidence"]
    second_confidence = top_3[1]["confidence"]
    confidence_margin = confidence - second_confidence

    uncertain = (
        confidence < confidence_threshold
        or confidence_margin < margin_threshold
    )

    return {
        "image_path": str(Path(image_path)),
        "predicted_style": top_3[0]["style"],
        "predicted_style_display": top_3[0]["display_style"],
        "confidence": confidence,
        "second_style": top_3[1]["style"],
        "second_style_display": top_3[1]["display_style"],
        "second_confidence": second_confidence,
        "confidence_margin": confidence_margin,
        "uncertain": uncertain,
        "certainty_status": (
            "LOW CONFIDENCE — manual review recommended"
            if uncertain
            else "CONFIDENT"
        ),
        "top_3": top_3,
    }


print("✅ Official V2 image preprocessing ready")
print("✅ Five-class classifier inference ready")
print("✅ Confidence and prediction-margin checks ready")


✅ Official V2 image preprocessing ready
✅ Five-class classifier inference ready
✅ Confidence and prediction-margin checks ready


In [5]:
# CELL 5 — RETRIEVAL, STYLE FILTERING, AND CROSS-ENCODER RERANKING

import re


def clean_passage(text):
    text = str(text or "")
    text = re.sub(
        r"(?<=[A-Za-z])-\s*(?:\r?\n)+\s*(?=[a-z])",
        "",
        text,
    )
    return re.sub(r"\s+", " ", text).strip()


def styles_mentioned_in_question(question):
    normalized_question = (
        str(question)
        .lower()
        .replace("-", " ")
        .replace("_", " ")
    )

    mentioned = []

    for style_name in CLASS_NAMES:
        readable_name = style_name.replace("_", " ")

        if readable_name in normalized_question:
            mentioned.append(style_name)

    return mentioned


def retrieve_and_rerank_evidence(
    query,
    predicted_style=None,
    retrieval_k=8,
    final_k=3,
):
    """
    Retrieve from the existing Chroma collection, then rerank with the
    existing cross-encoder. The database remains read-only.
    """
    query = str(query).strip()

    if not query:
        raise ValueError("A retrieval query is required.")

    mentioned_styles = styles_mentioned_in_question(
        query
    )

    # Keep image-led questions grounded in the predicted style.
    # Remove the filter for explicit cross-style comparisons.
    style_filter = None

    if (
        predicted_style in CLASS_NAMES
        and not any(
            style != predicted_style
            for style in mentioned_styles
        )
    ):
        style_filter = {"style": predicted_style}

    search_arguments = {
        "query": query,
        "k": retrieval_k,
    }

    if style_filter is not None:
        search_arguments["filter"] = style_filter

    retrieved_results = (
        vector_database.similarity_search_with_score(
            **search_arguments
        )
    )

    # A safe fallback allows general evidence if a metadata filter
    # unexpectedly has no matching records.
    if not retrieved_results and style_filter is not None:
        retrieved_results = (
            vector_database.similarity_search_with_score(
                query,
                k=retrieval_k,
            )
        )

    if not retrieved_results:
        raise RuntimeError(
            "No evidence was retrieved from the RAG database."
        )

    candidate_passages = [
        clean_passage(document.page_content)
        for document, _ in retrieved_results
    ]

    reranker_pairs = [
        (query, passage)
        for passage in candidate_passages
    ]

    reranker_scores = np.asarray(
        reranker.predict(
            reranker_pairs,
            show_progress_bar=False,
        )
    ).reshape(-1)

    ranked_indices = np.argsort(
        reranker_scores
    )[::-1][:final_k]

    evidence_records = []
    seen_record_keys = set()

    for index in ranked_indices:
        document, retrieval_distance = (
            retrieved_results[int(index)]
        )

        metadata = dict(document.metadata or {})
        passage = clean_passage(
            document.page_content
        )

        record_key = (
            metadata.get("chunk_id")
            or metadata.get("title")
            or passage[:120]
        )

        if record_key in seen_record_keys:
            continue

        seen_record_keys.add(record_key)

        evidence_records.append(
            {
                "rank": len(evidence_records) + 1,
                "citation": (
                    f"[S{len(evidence_records) + 1}]"
                ),
                "title": metadata.get(
                    "title",
                    "Architectural reference",
                ),
                "style": metadata.get(
                    "style",
                    "unknown",
                ),
                "source": (
                    metadata.get("source")
                    or metadata.get("source_url")
                    or "Not available"
                ),
                "source_organisation": (
                    metadata.get("source_organisation")
                    or metadata.get("source_organization")
                    or metadata.get("organisation")
                    or metadata.get("organization")
                    or "Not available"
                ),
                "chunk_id": metadata.get(
                    "chunk_id",
                    "Not available",
                ),
                "passage": passage,
                "retrieval_distance": float(
                    retrieval_distance
                ),
                "reranker_score": float(
                    reranker_scores[int(index)]
                ),
            }
        )

    if not evidence_records:
        raise RuntimeError(
            "Retrieved evidence could not be prepared."
        )

    return evidence_records


def format_evidence_context(evidence_records):
    context_sections = []

    for record in evidence_records:
        style_display = DISPLAY_NAMES.get(
            record["style"],
            str(record["style"])
            .replace("_", " ")
            .title(),
        )

        context_sections.append(
            "\n".join(
                [
                    (
                        f'{record["citation"]} '
                        f'Title: {record["title"]}'
                    ),
                    f"Architectural style: {style_display}",
                    (
                        "Source organisation: "
                        f'{record["source_organisation"]}'
                    ),
                    f'Source: {record["source"]}',
                    f'Passage: {record["passage"]}',
                ]
            )
        )

    return "\n\n".join(context_sections)


print("✅ Chroma evidence retrieval ready")
print("✅ Prediction-aware style filtering ready")
print("✅ Cross-style comparison retrieval ready")
print("✅ Cross-encoder reranking ready")


✅ Chroma evidence retrieval ready
✅ Prediction-aware style filtering ready
✅ Cross-style comparison retrieval ready
✅ Cross-encoder reranking ready


In [6]:
# CELL 6 — LANGCHAIN CONVERSATIONAL RAG CHAIN WITH MEMORY

from langchain_core.chat_history import (
    InMemoryChatMessageHistory,
)
from langchain_core.output_parsers import (
    StrOutputParser,
)
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.runnables import (
    RunnableLambda,
)
from langchain_core.runnables.history import (
    RunnableWithMessageHistory,
)


SYSTEM_PROMPT = """
You are HeritageLens, an educational architectural-style assistant.

Rules:
1. Use only the supplied reference evidence for architectural facts.
2. Cite factual claims with [S1], [S2], or [S3].
3. Treat the classifier output as a prediction, not verified identity.
4. Never claim that you personally inspected a visual feature in the
   uploaded image. You receive classifier results, not image pixels.
5. If the prediction is low confidence, state that manual review is
   recommended.
6. If evidence is insufficient, say so clearly instead of guessing.
7. Use the conversation history to understand follow-up questions.
8. Be concise, clear, and suitable for a university demonstration.
""".strip()


rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        MessagesPlaceholder(
            variable_name="chat_history"
        ),
        (
            "human",
            """
Classifier context
- Predicted style: {predicted_style_display}
- Confidence: {confidence_text}
- Certainty: {certainty_status}
- Top predictions: {top_predictions_text}

User question
{question}

Retrieved and reranked evidence
{evidence_context}

Answer the question using the evidence and inline source citations.
""".strip(),
        ),
    ]
)

generation_chain = (
    rag_prompt
    | llm
    | StrOutputParser()
)


def message_text(message):
    content = getattr(message, "content", "")

    if isinstance(content, str):
        return content

    return str(content)


def prepare_rag_payload(inputs):
    question = str(
        inputs.get("question", "")
    ).strip()

    if not question:
        raise ValueError("Please enter a question.")

    chat_history = inputs.get(
        "chat_history",
        [],
    )

    history_excerpt = " ".join(
        message_text(message)
        for message in chat_history[-4:]
    )

    retrieval_query = " ".join(
        item
        for item in [
            history_excerpt,
            question,
        ]
        if item
    )

    predicted_style = inputs.get(
        "predicted_style"
    )

    evidence_records = (
        retrieve_and_rerank_evidence(
            query=retrieval_query,
            predicted_style=predicted_style,
            retrieval_k=8,
            final_k=3,
        )
    )

    return {
        "question": question,
        "chat_history": chat_history,
        "predicted_style_display": (
            inputs.get(
                "predicted_style_display"
            )
            or "No image prediction available"
        ),
        "confidence_text": inputs.get(
            "confidence_text",
            "Not available",
        ),
        "certainty_status": inputs.get(
            "certainty_status",
            "GENERAL QUESTION",
        ),
        "top_predictions_text": inputs.get(
            "top_predictions_text",
            "Not available",
        ),
        "evidence_context": (
            format_evidence_context(
                evidence_records
            )
        ),
        "evidence_records": evidence_records,
    }


def generate_answer_bundle(payload):
    answer = generation_chain.invoke(
        payload
    )

    return {
        "answer": str(answer).strip(),
        "evidence_records": payload[
            "evidence_records"
        ],
    }


rag_core_chain = (
    RunnableLambda(prepare_rag_payload)
    | RunnableLambda(generate_answer_bundle)
)

conversation_store = {}


def get_session_history(session_id):
    session_id = str(session_id)

    if session_id not in conversation_store:
        conversation_store[session_id] = (
            InMemoryChatMessageHistory()
        )

    return conversation_store[session_id]


conversational_rag_chain = RunnableWithMessageHistory(
    rag_core_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
    output_messages_key="answer",
)


def top_predictions_as_text(classification):
    return "; ".join(
        (
            f'{item["display_style"]}: '
            f'{item["confidence"]:.2%}'
        )
        for item in classification.get(
            "top_3",
            [],
        )
    ) or "Not available"


def invoke_conversational_rag(
    question,
    session_id,
    classification=None,
):
    classification = classification or {}

    result = conversational_rag_chain.invoke(
        {
            "question": question,
            "predicted_style": (
                classification.get(
                    "predicted_style"
                )
            ),
            "predicted_style_display": (
                classification.get(
                    "predicted_style_display"
                )
            ),
            "confidence_text": (
                f'{classification["confidence"]:.2%}'
                if "confidence" in classification
                else "Not available"
            ),
            "certainty_status": (
                classification.get(
                    "certainty_status",
                    "GENERAL QUESTION",
                )
            ),
            "top_predictions_text": (
                top_predictions_as_text(
                    classification
                )
            ),
        },
        config={
            "configurable": {
                "session_id": str(session_id)
            }
        },
    )

    return result


def clear_session_history(session_id):
    conversation_store.pop(
        str(session_id),
        None,
    )


print("✅ LangChain prompt and generation chain ready")
print("✅ Retrieval and reranking included in the RAG chain")
print("✅ Per-session conversational memory ready")
print("✅ Follow-up questions ready")
print("✅ Inline source citation rules ready")


✅ LangChain prompt and generation chain ready
✅ Retrieval and reranking included in the RAG chain
✅ Per-session conversational memory ready
✅ Follow-up questions ready
✅ Inline source citation rules ready


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [7]:
# CELL 7 — GRADIO APPLICATION LOGIC AND SAFE SOURCE RENDERING

import html
import traceback
from urllib.parse import urlparse

import gradio as gr


EMPTY_PREDICTION = """
<div class="card empty-card">
    <p class="eyebrow">V2 CLASSIFIER</p>
    <h2>Your prediction will appear here</h2>
    <p>Upload a clear exterior image and select Analyse architecture.</p>
</div>
"""

EMPTY_PROBABILITIES = """
<div class="card">
    <p class="eyebrow">TOP STYLE PROBABILITIES</p>
    <p class="muted">The three strongest predictions will appear here.</p>
</div>
"""

EMPTY_SOURCES = """
<div class="card">
    <p class="eyebrow">RERANKED SOURCES</p>
    <p class="muted">
        Retrieved evidence and source links will appear here.
    </p>
</div>
"""


def safe_external_url(value):
    value = str(value or "").strip()

    try:
        parsed = urlparse(value)

        if parsed.scheme in {"http", "https"} and parsed.netloc:
            return value
    except Exception:
        pass

    return ""


def render_prediction(classification):
    style = html.escape(
        classification[
            "predicted_style_display"
        ]
    )

    confidence = classification[
        "confidence"
    ]

    second_style = html.escape(
        classification[
            "second_style_display"
        ]
    )

    second_confidence = classification[
        "second_confidence"
    ]

    margin = classification[
        "confidence_margin"
    ]

    if classification["uncertain"]:
        status_class = "status warning"
        status_title = "Manual review recommended"
        status_text = (
            "Confidence or the margin between the two leading "
            "predictions is below the accepted threshold."
        )
    else:
        status_class = "status success"
        status_title = "Confident prediction"
        status_text = (
            "The prediction passed the configured confidence "
            "and margin checks."
        )

    return f"""
    <div class="card prediction-card">
        <p class="eyebrow">OFFICIAL V2 PREDICTION</p>
        <div class="prediction-heading">
            <div>
                <h2>{style}</h2>
                <p class="muted">Predicted architectural style</p>
            </div>
            <div class="confidence-badge">
                <strong>{confidence:.1%}</strong>
                <span>confidence</span>
            </div>
        </div>
        <div class="{status_class}">
            <strong>{html.escape(status_title)}</strong>
            <span>{html.escape(status_text)}</span>
        </div>
        <div class="metric-grid">
            <div class="metric">
                <span>SECOND PREDICTION</span>
                <strong>{second_style}</strong>
                <small>{second_confidence:.2%}</small>
            </div>
            <div class="metric">
                <span>CONFIDENCE MARGIN</span>
                <strong>{margin:.2%}</strong>
                <small>Top 1 minus Top 2</small>
            </div>
        </div>
    </div>
    """


def render_probabilities(classification):
    rows = []

    for item in classification["top_3"]:
        style = html.escape(
            item["display_style"]
        )
        probability = item["confidence"]
        width = max(
            2.0,
            min(100.0, probability * 100),
        )

        rows.append(
            f"""
            <div class="probability-row">
                <div class="probability-label">
                    <span>{style}</span>
                    <strong>{probability:.2%}</strong>
                </div>
                <div class="probability-track">
                    <div
                        class="probability-fill"
                        style="width: {width:.2f}%"
                    ></div>
                </div>
            </div>
            """
        )

    return f"""
    <div class="card">
        <p class="eyebrow">TOP STYLE PROBABILITIES</p>
        {''.join(rows)}
    </div>
    """


def render_sources(evidence_records):
    source_cards = []

    for record in evidence_records:
        citation = html.escape(
            record["citation"]
        )
        title = html.escape(
            str(record["title"])
        )
        style = html.escape(
            DISPLAY_NAMES.get(
                record["style"],
                str(record["style"])
                .replace("_", " ")
                .title(),
            )
        )
        organisation = html.escape(
            str(record["source_organisation"])
        )
        passage = html.escape(
            str(record["passage"])
        )
        source_url = safe_external_url(
            record["source"]
        )

        source_link = (
            (
                f'<a href="{html.escape(source_url, quote=True)}" '
                'target="_blank" rel="noopener noreferrer">'
                "Open original source ↗</a>"
            )
            if source_url
            else (
                '<span class="source-unavailable">'
                "Source URL unavailable</span>"
            )
        )

        source_cards.append(
            f"""
            <article class="source-card">
                <div class="source-topline">
                    <span class="citation-chip">{citation}</span>
                    <span class="style-chip">{style}</span>
                </div>
                <h3>{title}</h3>
                <p class="source-org">{organisation}</p>
                <p class="source-passage">{passage}</p>
                <div class="source-footer">
                    {source_link}
                    <span>
                        Reranker:
                        {record["reranker_score"]:.4f}
                    </span>
                </div>
            </article>
            """
        )

    return f"""
    <div class="card sources-card">
        <p class="eyebrow">RERANKED SOURCES</p>
        <h2>Evidence used by the answer</h2>
        <div class="source-list">
            {''.join(source_cards)}
        </div>
    </div>
    """


def sources_as_markdown(evidence_records):
    lines = ["", "**Sources used**"]

    for record in evidence_records:
        label = (
            f'{record["citation"]} '
            f'{record["title"]}'
        )
        source_url = safe_external_url(
            record["source"]
        )

        if source_url:
            lines.append(
                f"- [{label}]({source_url})"
            )
        else:
            lines.append(f"- {label}")

    return "\n".join(lines)


def render_application_error(title, message):
    return f"""
    <div class="card application-error">
        <p class="eyebrow">ACTION NEEDED</p>
        <h2>{html.escape(str(title))}</h2>
        <p>{html.escape(str(message))}</p>
    </div>
    """


def append_assistant_message(chat_messages, message):
    messages = list(chat_messages or [])
    messages.append(
        {
            "role": "assistant",
            "content": str(message),
        }
    )
    return messages


def analyze_image_ui(
    image_path,
    chat_messages,
    session_id,
):
    if not image_path:
        message = "Please upload a building image first."
        return (
            render_application_error(
                "No image selected",
                message,
            ),
            EMPTY_PROBABILITIES,
            EMPTY_SOURCES,
            append_assistant_message(
                chat_messages,
                message,
            ),
            {},
        )

    try:
        # A new image starts a clean, image-specific conversation.
        clear_session_history(session_id)

        classification = (
            classify_architecture_image(
                image_path
            )
        )

        initial_question = (
            "Explain the classifier prediction using the retrieved "
            "architectural evidence. Describe the typical features "
            "of the predicted style, clearly state the confidence, "
            "and avoid claiming that the building identity is verified."
        )

        rag_result = invoke_conversational_rag(
            question=initial_question,
            session_id=session_id,
            classification=classification,
        )

        answer_markdown = rag_result["answer"]

        updated_messages = [
            {
                "role": "assistant",
                "content": answer_markdown,
            }
        ]

        return (
            render_prediction(classification),
            render_probabilities(classification),
            render_sources(
                rag_result["evidence_records"]
            ),
            updated_messages,
            classification,
        )
    except Exception as error:
        traceback.print_exc()
        message = (
            f"Analysis could not complete: {error}. "
            "The details are also printed below Cell 8 in Colab."
        )
        return (
            render_application_error(
                "Analysis unavailable",
                message,
            ),
            EMPTY_PROBABILITIES,
            EMPTY_SOURCES,
            append_assistant_message(
                chat_messages,
                message,
            ),
            {},
        )


def answer_follow_up_ui(
    question,
    chat_messages,
    classification,
    session_id,
):
    question = str(question or "").strip()
    chat_messages = list(
        chat_messages or []
    )

    if not question:
        return (
            "",
            append_assistant_message(
                chat_messages,
                "Please enter a question before selecting Ask HeritageLens.",
            ),
            EMPTY_SOURCES,
        )

    if not classification:
        return (
            "",
            append_assistant_message(
                chat_messages,
                "Please analyse a building image before asking a "
                "follow-up question.",
            ),
            EMPTY_SOURCES,
        )

    try:
        rag_result = invoke_conversational_rag(
            question=question,
            session_id=session_id,
            classification=classification,
        )

        chat_messages.extend(
            [
                {
                    "role": "user",
                    "content": question,
                },
                {
                    "role": "assistant",
                    "content": rag_result["answer"],
                },
            ]
        )

        return (
            "",
            chat_messages,
            render_sources(
                rag_result["evidence_records"]
            ),
        )
    except Exception as error:
        traceback.print_exc()
        chat_messages.extend(
            [
                {
                    "role": "user",
                    "content": question,
                },
                {
                    "role": "assistant",
                    "content": (
                        f"I could not answer this question: {error}. "
                        "The technical details are printed below Cell 8 "
                        "in Colab."
                    ),
                },
            ]
        )
        return (
            "",
            chat_messages,
            EMPTY_SOURCES,
        )


def reset_application(session_id):
    clear_session_history(session_id)

    return (
        None,
        EMPTY_PREDICTION,
        EMPTY_PROBABILITIES,
        EMPTY_SOURCES,
        [],
        {},
        str(uuid.uuid4()),
        "",
    )


print("✅ Gradio analysis callback ready")
print("✅ Follow-up conversation callback ready")
print("✅ Safe source-link rendering ready")
print("✅ Full session reset ready")


✅ Gradio analysis callback ready
✅ Follow-up conversation callback ready
✅ Safe source-link rendering ready
✅ Full session reset ready


In [ ]:
# CELL 8 — BUILD AND LAUNCH THE FINAL HERITAGELENS APPLICATION

heritagelens_css = """
:root {
    color-scheme: light;
    --ink: #172019;
    --muted: #667069;
    --cream: #f5f1e8;
    --surface: #fffdf8;
    --line: #ddd6c8;
    --gold: #a8762a;
    --green: #2f664b;
    --amber: #8a5a10;
}

body {
    background:
        radial-gradient(
            circle at 8% 0%,
            rgba(168, 118, 42, 0.12),
            transparent 30%
        ),
        var(--cream) !important;
}

.gradio-container {
    max-width: 1240px !important;
    margin: auto !important;
    padding: 26px 18px 54px !important;
    background: transparent !important;
    color: var(--ink) !important;
}

footer {
    display: none !important;
}

.hero {
    position: relative;
    overflow: hidden;
    padding: 46px;
    margin-bottom: 22px;
    color: white;
    background: #172019;
    border-radius: 28px;
    box-shadow: 0 22px 65px rgba(23, 32, 25, 0.16);
}

.hero::after {
    content: "";
    position: absolute;
    width: 360px;
    height: 360px;
    right: -120px;
    top: -175px;
    border: 1px solid rgba(255, 255, 255, 0.13);
    border-radius: 50%;
    box-shadow:
        0 0 0 55px rgba(255, 255, 255, 0.025),
        0 0 0 110px rgba(255, 255, 255, 0.018);
}

.brand {
    color: #dbb974;
    font-size: 12px;
    font-weight: 800;
    letter-spacing: 0.18em;
}

.hero h1 {
    max-width: 790px;
    margin: 28px 0 16px !important;
    color: white !important;
    font-family: Georgia, "Times New Roman", serif !important;
    font-size: clamp(40px, 6vw, 72px) !important;
    line-height: 0.98 !important;
    letter-spacing: -0.045em !important;
}

.hero p {
    max-width: 700px;
    color: #cbd1cc !important;
    font-size: 16px !important;
    line-height: 1.65 !important;
}

.style-list {
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
    margin-top: 24px;
}

.style-list span {
    padding: 8px 12px;
    color: #eee8dc;
    background: rgba(255, 255, 255, 0.055);
    border: 1px solid rgba(255, 255, 255, 0.16);
    border-radius: 999px;
    font-size: 12px;
}

.workspace {
    padding: 19px !important;
    background: rgba(255, 253, 248, 0.88) !important;
    border: 1px solid var(--line) !important;
    border-radius: 23px !important;
    box-shadow: 0 14px 45px rgba(45, 37, 25, 0.06);
}

.section-intro h2 {
    margin: 0 0 6px !important;
    color: var(--ink) !important;
    font-family: Georgia, "Times New Roman", serif !important;
    font-size: 28px !important;
}

.section-intro p,
.muted {
    color: var(--muted) !important;
}

#heritage-image {
    min-height: 390px !important;
    overflow: hidden !important;
    background: #f9f6ee !important;
    border: 1px dashed #b7a98f !important;
    border-radius: 17px !important;
}

#analyse-button,
#send-button {
    min-height: 48px !important;
    color: white !important;
    background: var(--ink) !important;
    border: 1px solid var(--ink) !important;
    border-radius: 12px !important;
    font-weight: 750 !important;
}

#clear-button {
    min-height: 48px !important;
    color: var(--ink) !important;
    background: transparent !important;
    border: 1px solid var(--line) !important;
    border-radius: 12px !important;
}

.card {
    padding: 24px;
    margin-bottom: 16px;
    color: var(--ink);
    background: var(--surface);
    border: 1px solid var(--line);
    border-radius: 20px;
    box-shadow: 0 8px 28px rgba(45, 37, 25, 0.045);
}

.empty-card {
    min-height: 220px;
    display: flex;
    flex-direction: column;
    justify-content: center;
}

.card h2,
.card h3 {
    color: var(--ink);
    font-family: Georgia, "Times New Roman", serif;
}

.card h2 {
    margin: 0 0 8px;
    font-size: 30px;
}

.eyebrow {
    margin: 0 0 10px !important;
    color: #714a13 !important;
    font-size: 10px !important;
    font-weight: 850 !important;
    letter-spacing: 0.15em !important;
}

.prediction-heading {
    display: flex;
    justify-content: space-between;
    gap: 18px;
    align-items: flex-start;
}

.confidence-badge {
    width: 108px;
    height: 108px;
    flex: 0 0 108px;
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
    background: #f5eddd;
    border: 7px solid #d9b977;
    border-radius: 50%;
}

.confidence-badge strong {
    color: var(--ink);
    font-size: 21px;
}

.confidence-badge span {
    color: var(--muted);
    font-size: 10px;
}

.status {
    display: flex;
    flex-direction: column;
    gap: 4px;
    padding: 13px 15px;
    margin: 20px 0 16px;
    border-radius: 13px;
}

.status.success {
    color: #244c38;
    background: #e3efe7;
    border: 1px solid #aac8b5;
}

.status.warning {
    color: #633e06;
    background: #fce9bd;
    border: 1px solid #d6aa52;
}

.status span {
    font-size: 12px;
    line-height: 1.45;
}

.metric-grid {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 11px;
}

.metric {
    padding: 14px;
    background: #f6f2e9;
    border-radius: 13px;
}

.metric span,
.metric small {
    display: block;
    color: var(--muted);
    font-size: 10px;
}

.metric strong {
    display: block;
    margin: 5px 0 2px;
    color: var(--ink);
}

.probability-row {
    margin-top: 16px;
}

.probability-label {
    display: flex;
    justify-content: space-between;
    margin-bottom: 6px;
    color: var(--ink);
    font-size: 13px;
}

.probability-track {
    height: 9px;
    overflow: hidden;
    background: #e9e2d6;
    border-radius: 99px;
}

.probability-fill {
    height: 100%;
    background: linear-gradient(90deg, #86591d, #d0a45c);
    border-radius: 99px;
}

.sources-card {
    margin-top: 18px;
}

.source-list {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 12px;
    margin-top: 17px;
}

.source-card {
    display: flex;
    flex-direction: column;
    padding: 17px;
    background: #f7f3ea;
    border: 1px solid #e2dacd;
    border-radius: 15px;
}

.source-topline,
.source-footer {
    display: flex;
    justify-content: space-between;
    gap: 10px;
    align-items: center;
}

.citation-chip,
.style-chip {
    padding: 6px 9px;
    border-radius: 999px;
    font-size: 10px;
    font-weight: 800;
}

.citation-chip {
    color: white;
    background: var(--ink);
}

.style-chip {
    color: #714a13;
    background: #ede2cd;
}

.source-card h3 {
    margin: 15px 0 5px;
    font-size: 19px;
}

.source-org {
    margin: 0 0 12px;
    color: #714a13;
    font-size: 11px;
    font-weight: 750;
}

.source-passage {
    flex: 1;
    color: #414a44;
    font-size: 12px;
    line-height: 1.58;
}

.source-footer {
    padding-top: 13px;
    margin-top: 12px;
    border-top: 1px solid #e2dacd;
    color: var(--muted);
    font-size: 10px;
}

.source-footer a {
    color: #714a13;
    font-weight: 750;
    text-decoration: none;
}

.source-unavailable {
    color: var(--muted);
}

.chat-heading {
    margin-top: 22px;
}

.chat-heading h2 {
    color: var(--ink) !important;
    font-family: Georgia, "Times New Roman", serif !important;
    font-size: 30px !important;
}

#heritage-chatbot {
    background: var(--surface) !important;
    border: 1px solid var(--line) !important;
    border-radius: 20px !important;
}

.disclaimer {
    max-width: 880px;
    margin: 25px auto 0;
    color: var(--muted) !important;
    text-align: center;
    font-size: 11px;
    line-height: 1.65;
}

@media (max-width: 900px) {
    .source-list {
        grid-template-columns: 1fr;
    }
}

@media (max-width: 700px) {
    .gradio-container {
        padding: 12px 11px 34px !important;
    }

    .hero {
        padding: 31px 23px;
        border-radius: 21px;
    }

    .prediction-heading {
        flex-direction: column;
    }

    .metric-grid {
        grid-template-columns: 1fr;
    }
}
"""

# Production UI layer
#
# Gradio can select dark tokens from the browser preference even when the
# application is designed as a light experience. These overrides define both
# light and dark component tokens explicitly and then style each interactive
# area so custom HTML and native Gradio components use one visual system.
heritagelens_css += """
html,
body,
body.dark,
#root,
.gradio-container {
    color-scheme: light !important;
    background: #f2f5f7 !important;
}

.gradio-container {
    --body-background-fill: #f2f5f7;
    --body-background-fill-dark: #f2f5f7;
    --body-text-color: #102a36;
    --body-text-color-dark: #102a36;
    --body-text-color-subdued: #526771;
    --body-text-color-subdued-dark: #526771;
    --block-background-fill: #ffffff;
    --block-background-fill-dark: #ffffff;
    --block-border-color: #d6e0e4;
    --block-border-color-dark: #d6e0e4;
    --block-label-text-color: #29434f;
    --block-label-text-color-dark: #29434f;
    --block-title-text-color: #102a36;
    --block-title-text-color-dark: #102a36;
    --input-background-fill: #ffffff;
    --input-background-fill-dark: #ffffff;
    --input-border-color: #cfc7b8;
    --input-border-color-dark: #cfc7b8;
    --input-placeholder-color: #69736c;
    --input-placeholder-color-dark: #69736c;
    --button-secondary-background-fill: #ffffff;
    --button-secondary-background-fill-dark: #ffffff;
    --button-secondary-text-color: #102a36;
    --button-secondary-text-color-dark: #102a36;
    --button-secondary-border-color: #c8d6dc;
    --button-secondary-border-color-dark: #c8d6dc;
    --panel-background-fill: #ffffff;
    --panel-background-fill-dark: #ffffff;
    max-width: 1380px !important;
    padding: 22px 24px 52px !important;
    color: #102a36 !important;
    font-family:
        Inter, ui-sans-serif, system-ui, -apple-system,
        BlinkMacSystemFont, "Segoe UI", sans-serif !important;
}

.gradio-container > .main,
.gradio-container .main,
.gradio-container .contain {
    background: transparent !important;
}

.hero {
    min-height: 300px;
    display: flex;
    flex-direction: column;
    justify-content: center;
    padding: 44px 50px;
    margin-bottom: 24px;
    background:
        radial-gradient(
            circle at 88% 15%,
            rgba(103, 205, 196, 0.22),
            transparent 27%
        ),
        linear-gradient(118deg, #0d2d3b 0%, #114b55 60%, #17636a 100%);
    border: 1px solid rgba(255, 255, 255, 0.08);
    border-radius: 28px;
    box-shadow: 0 24px 60px rgba(24, 35, 28, 0.16);
}

.hero::before {
    content: "FINAL V3  •  GRADIO 6.20";
    position: absolute;
    right: 44px;
    bottom: 34px;
    z-index: 1;
    padding: 9px 12px;
    color: #fff1cf;
    background: rgba(8, 15, 11, 0.24);
    border: 1px solid rgba(255, 255, 255, 0.12);
    border-radius: 999px;
    font-size: 10px;
    font-weight: 800;
    letter-spacing: 0.13em;
}

.hero > * {
    position: relative;
    z-index: 2;
}

.hero h1 {
    max-width: 900px;
    margin: 22px 0 14px !important;
    font-size: clamp(42px, 5vw, 68px) !important;
    line-height: 1.02 !important;
}

.hero p {
    max-width: 760px;
    color: #d3dad5 !important;
    font-size: 16px !important;
}

.brand {
    color: #ffd38a !important;
}

.style-list span {
    color: #f4efe5 !important;
    background: rgba(255, 255, 255, 0.07) !important;
}

.workspace {
    padding: 22px !important;
    background: #ffffff !important;
    border: 1px solid #d6e0e4 !important;
    border-radius: 22px !important;
    box-shadow: 0 12px 34px rgba(41, 36, 27, 0.07) !important;
}

.section-intro {
    margin-bottom: 16px;
}

.section-intro h2 {
    color: #102a36 !important;
    font-size: 30px !important;
    line-height: 1.15 !important;
}

.section-intro p,
.section-intro p *,
.muted,
.muted * {
    color: #526771 !important;
    opacity: 1 !important;
}

#heritage-image {
    min-height: 430px !important;
    background: #f7f3eb !important;
    border: 1px dashed #a99b82 !important;
    border-radius: 16px !important;
}

#heritage-image,
#heritage-image *,
#heritage-image label,
#heritage-image p,
#heritage-image span,
#heritage-image button {
    color: #29342d !important;
    opacity: 1 !important;
}

#heritage-image button {
    background: #fffdfa !important;
    border-color: #cfc7b8 !important;
}

#analyse-button,
#send-button {
    min-height: 50px !important;
    color: #ffffff !important;
    background: #0f5861 !important;
    border-color: #0f5861 !important;
    border-radius: 12px !important;
    box-shadow: 0 7px 18px rgba(25, 39, 30, 0.14) !important;
    font-size: 14px !important;
    font-weight: 750 !important;
}

#analyse-button:hover,
#send-button:hover {
    color: #ffffff !important;
    background: #0b454d !important;
    border-color: #0b454d !important;
}

#clear-button {
    min-height: 50px !important;
    color: #243028 !important;
    background: #fffdfa !important;
    border: 1px solid #cfc7b8 !important;
    border-radius: 12px !important;
    font-size: 14px !important;
    font-weight: 700 !important;
}

#clear-button:hover {
    background: #f2ede3 !important;
}

.card {
    padding: 25px;
    margin-bottom: 16px;
    color: #102a36 !important;
    background: #ffffff !important;
    border: 1px solid #d6e0e4 !important;
    border-radius: 20px;
    box-shadow: 0 10px 30px rgba(41, 36, 27, 0.055);
}

.card *,
.prediction-card *,
.sources-card * {
    opacity: 1;
}

.card h2,
.card h3,
.prediction-heading h2,
.source-card h3 {
    color: #102a36 !important;
}

.eyebrow,
.eyebrow *,
.source-org,
.style-chip {
    color: #9a5518 !important;
}

.prediction-card {
    min-height: 330px;
}

.prediction-heading .muted {
    color: #5b665f !important;
}

.confidence-badge {
    background: #fff4df !important;
    border-color: #f0b85d !important;
    box-shadow: inset 0 0 0 1px rgba(125, 79, 17, 0.08);
}

.confidence-badge strong {
    color: #102a36 !important;
    font-size: 24px !important;
}

.confidence-badge span {
    color: #536057 !important;
    opacity: 1 !important;
}

.status.success {
    color: #173d2a !important;
    background: #dfeee5 !important;
    border: 1px solid #9fc4ad !important;
}

.status.warning {
    color: #573703 !important;
    background: #fbe6b6 !important;
    border: 1px solid #d1a044 !important;
}

.status,
.status *,
.status strong,
.status span {
    color: inherit !important;
    opacity: 1 !important;
}

.status strong {
    font-size: 14px !important;
}

.status span {
    font-size: 12px !important;
}

.application-error {
    min-height: 210px;
    border-color: #e3b8b8 !important;
    background: #fff7f7 !important;
}

.application-error h2 {
    color: #7a1f28 !important;
}

.application-error p:not(.eyebrow) {
    color: #5d3035 !important;
    line-height: 1.6;
}

.metric {
    background: #f3f7f8 !important;
    border: 1px solid #dbe5e8;
}

.metric span,
.metric small {
    color: #59645d !important;
    opacity: 1 !important;
}

.metric strong {
    color: #102a36 !important;
    font-size: 16px !important;
}

.probability-label,
.probability-label *,
.probability-label span,
.probability-label strong {
    color: #27322b !important;
    opacity: 1 !important;
}

.probability-label strong {
    font-variant-numeric: tabular-nums;
}

.probability-track {
    background: #e4ddd0 !important;
}

.probability-fill {
    background: linear-gradient(90deg, #0f5861, #4aa49e) !important;
}

.sources-card {
    margin-top: 22px;
}

.sources-card > h2 {
    margin-bottom: 5px;
}

.source-list {
    gap: 16px;
    margin-top: 20px;
}

.source-card {
    min-width: 0;
    padding: 20px;
    color: #243b46 !important;
    background: #f5f8f9 !important;
    border: 1px solid #d8e3e7 !important;
    border-radius: 16px;
}

.source-card h3 {
    min-height: 48px;
    margin: 16px 0 7px;
    font-size: 19px;
    line-height: 1.28;
}

.citation-chip {
    color: #ffffff !important;
    background: #0f5861 !important;
}

.style-chip {
    color: #834710 !important;
    background: #fff0d4 !important;
}

.source-org {
    color: #6b4513 !important;
    font-size: 12px !important;
}

.source-passage {
    min-height: 215px;
    max-height: 215px;
    padding-right: 7px;
    overflow-y: auto;
    color: #3e4942 !important;
    font-size: 13px !important;
    line-height: 1.62 !important;
    scrollbar-color: #c5b89f transparent;
    scrollbar-width: thin;
}

.source-footer,
.source-footer *,
.source-footer span {
    color: #56625a !important;
    opacity: 1 !important;
}

.source-footer a {
    color: #68420d !important;
    font-weight: 800 !important;
}

.source-unavailable {
    color: #59655d !important;
}

.chat-heading {
    margin: 28px 0 13px;
    padding: 0 3px;
}

.chat-heading h2 {
    margin: 0 0 6px !important;
    color: #172019 !important;
    font-size: 32px !important;
    line-height: 1.15 !important;
}

.chat-heading p {
    color: #59655d !important;
    opacity: 1 !important;
}

#heritage-chatbot {
    min-height: 390px !important;
    overflow: hidden !important;
    color: #102a36 !important;
    background: #ffffff !important;
    border: 1px solid #d6e0e4 !important;
    border-radius: 18px !important;
    box-shadow: 0 10px 30px rgba(41, 36, 27, 0.055) !important;
}

#heritage-chatbot .message,
#heritage-chatbot .message.bot,
#heritage-chatbot [data-testid="bot"] {
    color: #203843 !important;
    background: #f1f6f7 !important;
    border: 1px solid #d8e3e7 !important;
}

#heritage-chatbot .message.user,
#heritage-chatbot [data-testid="user"] {
    color: #ffffff !important;
    background: #0f5861 !important;
    border-color: #0f5861 !important;
}

#heritage-chatbot .message.bot *,
#heritage-chatbot [data-testid="bot"] *,
#heritage-chatbot .message.bot p,
#heritage-chatbot .message.bot li {
    color: #28332c !important;
    opacity: 1 !important;
}

#heritage-chatbot .message.user *,
#heritage-chatbot [data-testid="user"] * {
    color: #ffffff !important;
    opacity: 1 !important;
}

#heritage-chatbot a {
    color: #70480e !important;
    font-weight: 750 !important;
}

#heritage-chatbot button,
#heritage-chatbot button * {
    color: #4e5a52 !important;
}

#question-input,
#question-input > div,
#question-input label,
#question-input .container {
    color: #1b241e !important;
    background: #fffdfa !important;
    border-color: #cfc7b8 !important;
}

#question-input textarea,
#question-input input {
    color: #1b241e !important;
    background: #ffffff !important;
    caret-color: #1b241e !important;
    opacity: 1 !important;
}

#question-input textarea::placeholder,
#question-input input::placeholder {
    color: #6b756e !important;
    opacity: 1 !important;
}

.disclaimer {
    color: #59655d !important;
    opacity: 1 !important;
}

@media (max-width: 1000px) {
    .hero::before {
        display: none;
    }

    .source-list {
        grid-template-columns: 1fr;
    }

    .source-card h3,
    .source-passage {
        min-height: auto;
    }

    .source-passage {
        max-height: 210px;
    }
}

@media (max-width: 700px) {
    .gradio-container {
        padding: 10px 10px 36px !important;
    }

    .hero {
        min-height: auto;
        padding: 31px 24px;
        border-radius: 20px;
    }

    .hero h1 {
        font-size: 40px !important;
    }

    .workspace,
    .card {
        padding: 18px !important;
        border-radius: 16px !important;
    }

    #heritage-image {
        min-height: 330px !important;
    }

    .prediction-heading {
        flex-direction: row;
    }

    .confidence-badge {
        width: 92px;
        height: 92px;
        flex-basis: 92px;
        border-width: 6px;
    }

    .metric-grid {
        grid-template-columns: 1fr 1fr;
    }

    .chat-heading h2 {
        font-size: 27px !important;
    }

    #heritage-chatbot {
        min-height: 350px !important;
    }
}

@media (max-width: 470px) {
    .prediction-heading,
    .metric-grid {
        grid-template-columns: 1fr;
    }

    .prediction-heading {
        flex-direction: column;
    }

    .metric-grid {
        display: grid;
    }
}
"""


previous_app = globals().get(
    "heritagelens_app"
)

if previous_app is not None:
    try:
        previous_app.close()
        print(
            "Closed the previous HeritageLens interface."
        )
    except Exception:
        pass


with gr.Blocks(
    title="HeritageLens",
    theme=gr.themes.Base(),
    css=heritagelens_css,
) as heritagelens_app:
    session_state = gr.State(
        value=str(uuid.uuid4())
    )
    classification_state = gr.State(
        value={}
    )

    gr.HTML(
        """
        <section class="hero">
            <div class="brand">
                HERITAGELENS STUDIO / PRODUCTION BUILD V3
            </div>
            <h1>Read architecture through evidence.</h1>
            <p>
                One production workspace for the official V2
                classifier, reranked architectural sources, and a
                grounded conversation that remembers your questions.
            </p>
            <div class="style-list">
                <span>Romanesque</span>
                <span>Gothic</span>
                <span>Tudor Revival</span>
                <span>Georgian</span>
                <span>Art Deco</span>
            </div>
        </section>
        """
    )

    with gr.Row(equal_height=False):
        with gr.Column(
            scale=5,
            elem_classes="workspace",
        ):
            gr.HTML(
                """
                <div class="section-intro">
                    <h2>Upload a building</h2>
                    <p>
                        Use a clear exterior image with the main
                        architectural form visible.
                    </p>
                </div>
                """
            )

            image_input = gr.Image(
                type="filepath",
                image_mode="RGB",
                sources=["upload"],
                label="Architecture image",
                show_label=False,
                height=390,
                elem_id="heritage-image",
            )

            with gr.Row():
                analyze_button = gr.Button(
                    "Analyse architecture",
                    variant="primary",
                    elem_id="analyse-button",
                )

                clear_button = gr.Button(
                    "Clear everything",
                    elem_id="clear-button",
                )

        with gr.Column(scale=5):
            prediction_output = gr.HTML(
                EMPTY_PREDICTION
            )
            probability_output = gr.HTML(
                EMPTY_PROBABILITIES
            )

    sources_output = gr.HTML(
        EMPTY_SOURCES
    )

    gr.HTML(
        """
        <div class="chat-heading">
            <h2>Ask a follow-up question</h2>
            <p class="muted">
                The assistant remembers this session and retrieves
                new evidence for each question.
            </p>
        </div>
        """
    )

    chatbot = gr.Chatbot(
        label="HeritageLens conversation",
        show_label=False,
        height=390,
        elem_id="heritage-chatbot",
    )

    with gr.Row():
        question_input = gr.Textbox(
            placeholder=(
                "Example: How is this style different from Gothic?"
            ),
            label="Question",
            show_label=False,
            lines=2,
            scale=8,
            elem_id="question-input",
        )

        send_button = gr.Button(
            "Ask HeritageLens",
            variant="primary",
            elem_id="send-button",
            scale=2,
        )

    gr.HTML(
        """
        <div class="disclaimer">
            HeritageLens supports five architectural styles.
            The classifier provides a prediction, while the RAG
            assistant explains reference characteristics from the
            saved knowledge base. It does not verify a building's
            identity. Low-confidence results require manual review.
        </div>
        """
    )

    analyze_button.click(
        fn=analyze_image_ui,
        inputs=[
            image_input,
            chatbot,
            session_state,
        ],
        outputs=[
            prediction_output,
            probability_output,
            sources_output,
            chatbot,
            classification_state,
        ],
        api_name="analyze_architecture",
    )

    send_button.click(
        fn=answer_follow_up_ui,
        inputs=[
            question_input,
            chatbot,
            classification_state,
            session_state,
        ],
        outputs=[
            question_input,
            chatbot,
            sources_output,
        ],
        api_name="ask_follow_up",
    )

    question_input.submit(
        fn=answer_follow_up_ui,
        inputs=[
            question_input,
            chatbot,
            classification_state,
            session_state,
        ],
        outputs=[
            question_input,
            chatbot,
            sources_output,
        ],
    )

    clear_button.click(
        fn=reset_application,
        inputs=[session_state],
        outputs=[
            image_input,
            prediction_output,
            probability_output,
            sources_output,
            chatbot,
            classification_state,
            session_state,
            question_input,
        ],
    )


print("\nFINAL APPLICATION CHECK")
print("-" * 72)
print("✅ Official V2 image classifier")
print("✅ Existing 71-record Chroma database")
print("✅ MiniLM embeddings")
print("✅ Cross-encoder reranking")
print("✅ Gemini answer generation")
print("✅ LangChain conversational RAG chain")
print("✅ Per-session conversation memory")
print("✅ Follow-up questions")
print("✅ Inline citations and visible source evidence")
print("✅ Gradio application")
print("✅ No training, evaluation, or database rebuilding")


heritagelens_app.queue().launch(
    share=True,
    debug=True,
    show_error=True,
)



FINAL APPLICATION CHECK
------------------------------------------------------------------------
✅ Official V2 image classifier
✅ Existing 71-record Chroma database
✅ MiniLM embeddings
✅ Cross-encoder reranking
✅ Gemini answer generation
✅ LangChain conversational RAG chain
✅ Per-session conversation memory
✅ Follow-up questions
✅ Inline citations and visible source evidence
✅ Gradio application
✅ No training, evaluation, or database rebuilding
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Gradio public link generated for this Colab session (temporary)

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/l

## Expected successful result

After the final cell runs:

- Colab prints all final application checks with green ticks.
- Gradio returns a public sharing link.
- Uploading an image produces the V2 prediction and top-three probabilities.
- Three reranked source cards appear.
- Gemini produces a grounded explanation with `[S1]`, `[S2]`, and `[S3]`
  citations.
- Follow-up questions use the same image context and conversation history.
- **Clear everything** removes the image, results, sources, and memory.

The training notebook is not required and should remain untouched.
